# Module 3: Dopamine, Prediction Errors, and TD Learning

**Spinning Up in Active Inference** | Notebook 3 of 16

---

In 1997, Wolfram Schultz published recordings from dopamine neurons in monkeys that changed neuroscience. The firing patterns of these neurons did not encode reward itself — they encoded the **difference between expected and received reward**. This *reward prediction error* (RPE) turned out to be exactly the quantity computed by the TD learning algorithm, which had been proposed by Richard Sutton over a decade earlier.

This module covers:
1. The **Rescorla-Wagner model** — the simplest prediction error learning rule
2. **TD(0)** — the foundational algorithm that bridges psychology and RL
3. The **RPE hypothesis** — dopamine as a neural TD error signal
4. **Successor Representations** — a halfway house between model-free and model-based

Each of these has a parallel in Active Inference: prediction errors drive learning in both frameworks, but AIF minimizes surprise about *all observations*, not just reward.

**Prerequisites:** Modules 1-2.  
**Time:** ~60 minutes.

## 3.1 The Rescorla-Wagner Model (1972)

Before TD learning, the dominant model of associative learning was the **Rescorla-Wagner rule**:

$$\Delta V = \alpha (\lambda - V)$$

where:
- $V$ is the current associative strength (prediction of reward)
- $\lambda$ is the actual outcome (1 if reward, 0 if not)
- $\alpha$ is the learning rate
- $(\lambda - V)$ is the **prediction error** — the surprise

This captures the key insight: **learning is driven by surprise.** When the outcome matches the prediction ($\lambda = V$), no learning occurs. When there is a mismatch, the prediction is updated.

This explains:
- **Acquisition:** $V$ starts at 0, so $\Delta V = \alpha \lambda > 0$ — prediction grows
- **Extinction:** Remove reward ($\lambda = 0$), so $\Delta V = -\alpha V < 0$ — prediction shrinks
- **Blocking:** A fully predicted reward ($V \approx \lambda$) produces no prediction error, so a new cue paired with it cannot acquire associative strength

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate
from neuronav.agents.td_agents import TDQ, TDSR

import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_value_heatmap, plot_learning_curve, plot_rpe_trace,
    plot_matrix_heatmap, plot_agent_trajectory,
    dual_perspective_box, COLORS
)

np.random.seed(42)
print("Imports successful!")

In [ ]:
# ── Rescorla-Wagner: Classical Conditioning Simulation ────────────────────────

def rescorla_wagner(n_trials=50, alpha=0.1, reward_present=None):
    """Simulate Rescorla-Wagner learning for a classical conditioning task.
    
    Args:
        n_trials: Number of learning trials.
        alpha: Learning rate.
        reward_present: Boolean array indicating whether reward is present
            on each trial. If None, reward is always present.
    
    Returns:
        V_history: Associative strength over trials.
        pe_history: Prediction errors over trials.
    """
    if reward_present is None:
        reward_present = np.ones(n_trials, dtype=bool)
    
    V = 0.0
    V_history = [V]
    pe_history = []
    
    for t in range(n_trials):
        lam = 1.0 if reward_present[t] else 0.0
        pe = lam - V  # prediction error
        V = V + alpha * pe
        V_history.append(V)
        pe_history.append(pe)
    
    return V_history, pe_history

# Simulate acquisition (30 trials) then extinction (20 trials)
reward_schedule = np.concatenate([np.ones(30), np.zeros(20)]).astype(bool)
V_hist, pe_hist = rescorla_wagner(n_trials=50, alpha=0.15, reward_present=reward_schedule)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Associative strength
ax1.plot(V_hist, color=COLORS['rl'], linewidth=2.5)
ax1.axvline(x=30, color='gray', linestyle='--', alpha=0.7)
ax1.text(15, 0.85, 'Acquisition\n(reward present)', ha='center', fontsize=11)
ax1.text(40, 0.85, 'Extinction\n(no reward)', ha='center', fontsize=11)
ax1.set_ylabel('Associative Strength V', fontsize=12)
ax1.set_title('Rescorla-Wagner Model: Classical Conditioning', fontsize=14)
ax1.grid(True, alpha=0.3)

# Prediction errors
colors_pe = [COLORS['success'] if pe > 0 else COLORS['danger'] for pe in pe_hist]
ax2.bar(range(len(pe_hist)), pe_hist, color=colors_pe, alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.axvline(x=30, color='gray', linestyle='--', alpha=0.7)
ax2.set_xlabel('Trial', fontsize=12)
ax2.set_ylabel('Prediction Error (\u03BB - V)', fontsize=12)
ax2.set_title('Prediction Errors (cf. Dopamine)', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Notice the key pattern:
- **Early acquisition:** Large positive prediction errors (surprise at receiving reward)
- **Late acquisition:** Prediction errors shrink toward zero (reward is fully predicted)
- **Extinction onset:** Large *negative* prediction error (surprise at *not* receiving reward)
- **Late extinction:** Prediction errors return to zero (absence of reward is now predicted)

This is exactly the pattern Schultz observed in dopamine neurons.

## 3.2 Schultz's Dopamine Recordings (1997)

Wolfram Schultz recorded from dopamine neurons in the ventral tegmental area (VTA) of monkeys performing a classical conditioning task. His findings:

| Condition | Dopamine Response |
|-----------|------------------|
| **Unexpected reward** | Burst of firing (positive RPE) |
| **Fully predicted reward** | No change in firing (RPE = 0) |
| **Expected reward omitted** | Pause in firing (negative RPE) |
| **Conditioned stimulus (predicts reward)** | Burst (prediction signal shifts to cue) |

The crucial insight: dopamine neurons do not encode reward itself. They encode the **temporal difference error**:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is exactly the TD(0) update signal. The convergence between a computational algorithm (Sutton, 1988) and neural recordings (Schultz, 1997) remains one of the most celebrated results in computational neuroscience.

## 3.3 TD(0) Learning

**Temporal-Difference learning** combines the best of Monte Carlo (learning from experience) and dynamic programming (bootstrapping from estimated values):

$$V(s_t) \leftarrow V(s_t) + \alpha \left[ r_t + \gamma V(s_{t+1}) - V(s_t) \right]$$

The TD error $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ has three terms:
1. $r_t$: the actual reward received (what happened)
2. $\gamma V(s_{t+1})$: the estimated future value (what I now expect)
3. $V(s_t)$: what I previously expected (what I predicted)

The error is the difference between the **better estimate** $(r_t + \gamma V(s_{t+1}))$ and the **old estimate** $V(s_t)$.

For **Q-learning** (action values), the update becomes:

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \right]$$

In [ ]:
# ── Train TDQ on T-Maze and record RPEs ──────────────────────────────────────
# The T-maze is a classic paradigm: the agent starts at the base of the T
# and must choose left or right at the junction to find the reward.

env = GridEnv(template=GridTemplate.four_rooms, size=GridSize.small, obs_type=GridObservation.index)
obs = env.reset()
goal_pos = list(env.objects['rewards'].keys())[0]  # Save BEFORE training

state_size = env.grid_size ** 2
action_size = 4

agent = TDQ(state_size, action_size, lr=0.1, gamma=0.99, 
            poltype="softmax", beta=5.0)

n_episodes = 200
episode_rewards = []
all_rpes = []  # RPEs from all timesteps across all episodes
episode_rpes = []  # mean absolute RPE per episode

for ep in range(n_episodes):
    obs = env.reset()
    done = False
    total_reward = 0
    steps = 0
    ep_rpes = []

    while not done and steps < 500:
        action = agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        
        # Compute RPE before update
        # agent.Q has shape (n_actions, n_states)
        if done:
            rpe = reward - agent.Q[action, obs]
        else:
            rpe = reward + agent.gamma * np.max(agent.Q[:, next_obs]) - agent.Q[action, obs]
        ep_rpes.append(rpe)
        all_rpes.append(rpe)

        agent.update((obs, action, next_obs, reward, done))
        obs = next_obs
        total_reward += reward
        steps += 1

    episode_rewards.append(total_reward)
    episode_rpes.append(np.mean(np.abs(ep_rpes)) if ep_rpes else 0)

print(f"Training complete!")
print(f"Mean reward (last 20): {np.mean(episode_rewards[-20:]):.3f}")
print(f"Mean |RPE| (last 20): {np.mean(episode_rpes[-20:]):.4f}")

In [ ]:
# ── Plot RPE evolution over learning ─────────────────────────────────────────
# Like Schultz's dopamine data: RPEs start large and shrink as the agent
# learns to predict reward.

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Learning curve
plot_learning_curve(
    {"TDQ Agent": episode_rewards},
    ylabel="Episode Reward",
    title="Learning Curve",
    window=10,
    ax=ax1
)

# RPE magnitude over episodes
window = 10
smoothed_rpe = np.convolve(episode_rpes, np.ones(window)/window, mode='valid')
ax2.plot(smoothed_rpe, color=COLORS['rl'], linewidth=2)
ax2.fill_between(range(len(smoothed_rpe)), 0, smoothed_rpe, alpha=0.2, color=COLORS['rl'])
ax2.set_xlabel('Episode', fontsize=12)
ax2.set_ylabel('Mean |RPE|', fontsize=12)
ax2.set_title('Prediction Error Magnitude Decreases with Learning\n'
              '(Like Schultz\'s dopamine: RPEs shrink as reward becomes predicted)', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── RPE spike when reward location changes ───────────────────────────────────
# This simulates "reward omission" from Schultz's data:
# the agent expects reward at the old location, but it's gone.

# Run a few episodes with the trained agent, recording RPEs
pre_change_rpes = []
for _ in range(5):
    obs = env.reset()
    done = False
    steps = 0
    while not done and steps < 200:
        action = agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        # agent.Q has shape (n_actions, n_states)
        if done:
            rpe = reward - agent.Q[action, obs]
        else:
            rpe = reward + agent.gamma * np.max(agent.Q[:, next_obs]) - agent.Q[action, obs]
        pre_change_rpes.append(rpe)
        obs = next_obs
        steps += 1

# Now change the goal position (reward revaluation)
# Use goal_pos saved before training (rewards dict may be empty after training)
old_goal = goal_pos
env.objects['rewards'] = {(1, 1): 1.0}  # move goal to a new location

post_change_rpes = []
for _ in range(5):
    obs = env.reset()
    done = False
    steps = 0
    while not done and steps < 200:
        action = agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        # agent.Q has shape (n_actions, n_states)
        if done:
            rpe = reward - agent.Q[action, obs]
        else:
            rpe = reward + agent.gamma * np.max(agent.Q[:, next_obs]) - agent.Q[action, obs]
        post_change_rpes.append(rpe)
        obs = next_obs
        steps += 1

# Restore goal
env.objects['rewards'] = {old_goal: 1.0}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

plot_rpe_trace(
    pre_change_rpes[:80],
    title="RPEs Before Goal Change\n(reward is well-predicted, RPEs near zero)",
    ax=ax1
)

plot_rpe_trace(
    post_change_rpes[:80],
    title="RPEs After Goal Change\n(negative RPEs: expected reward is missing!)",
    ax=ax2
)

plt.tight_layout()
plt.show()
print("After the goal moves, the agent experiences large negative RPEs —")
print("just like Schultz's dopamine dip when expected reward is omitted.")

## 3.4 The Successor Representation (SR)

The **Successor Representation** (Dayan, 1993) is a middle ground between model-free and model-based RL. Instead of learning Q-values directly, the SR factorizes the value function into two components:

$$V(s) = \sum_{s'} M(s, s') \cdot R(s')$$

where $M(s, s')$ is the **successor matrix** — the expected discounted number of times the agent will visit state $s'$ starting from state $s$:

$$M(s, s') = \mathbb{E}\left[ \sum_{t=0}^{\infty} \gamma^t \mathbb{1}(s_t = s') \mid s_0 = s \right]$$

The key insight is that $M$ captures **"where can I get to from here?"** while $R$ captures **"what's there?"** This separation enables:

- **Instant reward revaluation:** If the reward location changes, just update $R$ — no need to relearn $M$
- **Transfer:** The same $M$ works for different reward functions
- **Neural evidence:** Place cells in the hippocampus have firing patterns consistent with SR columns

The SR is particularly relevant for Active Inference because it demonstrates that separating "world dynamics" from "preferences" can be computationally advantageous — exactly what AIF's generative model does with its B matrix (transitions) and C vector (preferences).

In [ ]:
# ── Train a TDSR agent and visualize the successor matrix ────────────────────

env = GridEnv(template=GridTemplate.four_rooms, size=GridSize.small, obs_type=GridObservation.index)
obs = env.reset()
goal_pos = list(env.objects['rewards'].keys())[0]  # Save BEFORE training

state_size = env.grid_size ** 2
action_size = 4

sr_agent = TDSR(state_size, action_size, lr=0.1, gamma=0.99,
                poltype="softmax", beta=5.0)

# Train for 200 episodes
sr_rewards = []
for ep in range(200):
    obs = env.reset()
    done = False
    total_reward = 0
    steps = 0
    while not done and steps < 500:
        action = sr_agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        sr_agent.update((obs, action, next_obs, reward, done))
        obs = next_obs
        total_reward += reward
        steps += 1
    sr_rewards.append(total_reward)

print(f"TDSR mean reward (last 20): {np.mean(sr_rewards[-20:]):.3f}")

In [ ]:
# ── Visualize the SR matrix ──────────────────────────────────────────────────
# Each row M[s, :] is a "map" of expected future visits from state s.
# This is the agent's internal representation of the environment's structure.

# Get the SR matrix (average over actions for visualization)
# sr_agent.M has shape: (n_actions, n_states, n_states)
M_avg = np.mean(sr_agent.M, axis=0)  # average over actions: (n_states, n_states)

# Pick a few starting states and show their SR "maps"
start_states = [0, state_size // 4, state_size // 2, state_size - 1]
fig, axes = plt.subplots(1, len(start_states), figsize=(5*len(start_states), 4.5))

for ax, s in zip(axes, start_states):
    sr_map = M_avg[s, :]
    pos = (s // env.grid_size, s % env.grid_size)
    plot_value_heatmap(
        sr_map, env.grid_size,
        title=f"SR from state {s}\n(position {pos})",
        cmap="Blues",
        agent_pos=pos,
        show_values=False,
        ax=ax
    )

fig.suptitle('Successor Representation: "Where can I get to from here?"', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── SR enables instant reward revaluation ────────────────────────────────────
# The value function V(s) = M(s,:) @ R can be recomputed with a new reward
# vector without retraining. This is Tolman's latent learning in action!

# Use goal_pos saved before training (rewards dict may be empty after training)
goal_state = goal_pos[0] * env.grid_size + goal_pos[1]

# Original reward vector
R_original = np.zeros(state_size)
R_original[goal_state] = 1.0

# New reward vector: reward is in a completely different location
new_goal = (1, 1)
new_goal_state = new_goal[0] * env.grid_size + new_goal[1]
R_new = np.zeros(state_size)
R_new[new_goal_state] = 1.0

# Compute value functions
# sr_agent.M has shape (n_actions, n_states, n_states)
M_avg = np.mean(sr_agent.M, axis=0)
V_original = M_avg @ R_original
V_new = M_avg @ R_new  # instant revaluation — no retraining!

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

plot_value_heatmap(
    V_original, env.grid_size,
    title=f"V(s) with original goal at {goal_pos}",
    cmap="inferno",
    goal_pos=goal_pos,
    show_values=False,
    ax=ax1
)

plot_value_heatmap(
    V_new, env.grid_size,
    title=f"V(s) with new goal at {new_goal}\n(instant revaluation via SR, no retraining!)",
    cmap="inferno",
    goal_pos=new_goal,
    show_values=False,
    ax=ax2
)

plt.tight_layout()
plt.show()
print("The SR separates 'where can I go?' (M) from 'what do I want?' (R).")
print("This is exactly what AIF's generative model does: B encodes dynamics, C encodes preferences.")

In [ ]:
# ── Dual Perspective: Prediction Errors in RL vs. AIF ────────────────────────

dual_perspective_box(
    rl_text=(
        "TD learning is driven by Reward Prediction Errors: "
        "\u03B4 = r + \u03B3V(s') - V(s). This signal maps onto phasic dopamine "
        "(Schultz 1997). The agent learns by minimizing RPEs — when the world "
        "is fully predicted, \u03B4 \u2192 0 and learning stops. The SR separates "
        "'where can I go' (M) from 'what do I want' (R), enabling revaluation."
    ),
    aif_text=(
        "Active Inference agents also learn from prediction errors — but over "
        "ALL observations, not just reward. The sensory prediction error "
        "\u03B5 = o - E_q[o] drives belief updates via variational free energy "
        "minimization. Surprise = -log P(o|s). The generative model's B matrix "
        "plays the role of M (dynamics) and C plays the role of R (preferences) — "
        "the same factorization as the SR, but derived from first principles."
    ),
    title="Prediction Errors: Dopamine (RL) vs. Surprise Minimization (AIF)"
)

In [ ]:
# ── Visualize the full SR matrix structure ───────────────────────────────────
# The SR matrix captures the topology of the environment.
# Block structure reveals the room layout in Four Rooms.

# Show a submatrix (full matrix is 121x121 for GridSize.small)
n_show = min(40, state_size)

fig, ax = plt.subplots(1, 1, figsize=(8, 7))
plot_matrix_heatmap(
    M_avg[:n_show, :n_show],
    title=f"Successor Representation Matrix M (first {n_show} states)\n"
          f"Block structure reveals the room connectivity",
    cmap="Blues",
    ax=ax,
    fmt=".1f"
)
ax.set_xlabel("Future state s'")
ax.set_ylabel("Current state s")
plt.tight_layout()
plt.show()

## 3.5 Exercises

### Exercise 1: Learning Rate Comparison
Compare TDQ agents with different learning rates ($\alpha = 0.01, 0.1, 0.5$). Which converges fastest? Which is most stable? How does this relate to the Rescorla-Wagner learning rate?

In [ ]:
# ── Exercise 1: Learning rate comparison ─────────────────────────────────────

learning_rates = [0.01, 0.1, 0.5]
lr_curves = {}

for lr in learning_rates:
    env_lr = GridEnv(template=GridTemplate.four_rooms, size=GridSize.small, obs_type=GridObservation.index)
    agent_lr = TDQ(state_size, action_size, lr=lr, gamma=0.99,
                   poltype="softmax", beta=5.0)
    rewards_lr = []
    for ep in range(200):
        obs = env_lr.reset()
        done = False
        total_reward = 0
        steps = 0
        while not done and steps < 500:
            action = agent_lr.sample_action(obs)
            next_obs, reward, done, info = env_lr.step(action)
            agent_lr.update((obs, action, next_obs, reward, done))
            obs = next_obs
            total_reward += reward
            steps += 1
        rewards_lr.append(total_reward)
    lr_curves[f"TDQ (lr={lr})"] = rewards_lr

plot_learning_curve(
    lr_curves,
    title="Effect of Learning Rate on TD Q-Learning",
    ylabel="Episode Reward",
    window=10
)
plt.tight_layout()
plt.show()

### Exercise 2: Reward Revaluation
Train a TDSR agent, then change the reward location. Show that the SR agent can instantly adapt (by recomputing $V = M \cdot R_{new}$) while a TDQ agent must relearn from scratch.

In [ ]:
# ── Exercise 2: Reward revaluation comparison ────────────────────────────────
# This is Tolman's latent learning experiment, computationally!

# Train both agents on original goal
env_reval = GridEnv(template=GridTemplate.four_rooms, size=GridSize.small, obs_type=GridObservation.index)
obs = env_reval.reset()
goal_pos_reval = list(env_reval.objects['rewards'].keys())[0]  # Save BEFORE training

q_agent = TDQ(state_size, action_size, lr=0.1, gamma=0.99, poltype="softmax", beta=5.0)
sr_agent_reval = TDSR(state_size, action_size, lr=0.1, gamma=0.99, poltype="softmax", beta=5.0)

# Phase 1: Train both on original goal
for ep in range(200):
    obs = env_reval.reset()
    done = False
    steps = 0
    while not done and steps < 500:
        action_q = q_agent.sample_action(obs)
        next_obs, reward, done, info = env_reval.step(action_q)
        q_agent.update((obs, action_q, next_obs, reward, done))
        sr_agent_reval.update((obs, action_q, next_obs, reward, done))
        obs = next_obs
        steps += 1

# Show the V functions side by side
# agent.Q has shape (n_actions, n_states)
V_q = np.max(q_agent.Q, axis=0)
# sr_agent_reval.M has shape (n_actions, n_states, n_states)
M_sr = np.mean(sr_agent_reval.M, axis=0)

# Use goal_pos_reval saved before training
goal_state = goal_pos_reval[0] * env.grid_size + goal_pos_reval[1]

# Create new reward at different location
R_old = np.zeros(state_size)
R_old[goal_state] = 1.0
R_shifted = np.zeros(state_size)
new_goal_idx = 5 * env.grid_size + 5  # position (5,5)
R_shifted[new_goal_idx] = 1.0

V_sr_new = M_sr @ R_shifted  # instant revaluation!

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

plot_value_heatmap(
    V_q, env.grid_size,
    title="TDQ Agent: Trained on original goal\n(must retrain from scratch for new goal)",
    cmap="inferno", show_values=False,
    goal_pos=goal_pos_reval, ax=ax1
)

new_pos = (5, 5)
plot_value_heatmap(
    V_sr_new, env.grid_size,
    title=f"TDSR Agent: Instantly revalued for new goal at {new_pos}\n(V = M @ R_new, no retraining!)",
    cmap="inferno", show_values=False,
    goal_pos=new_pos, ax=ax2
)

plt.tight_layout()
plt.show()

### Exercise 3: SR Matrix Structure
Compare the SR matrix before and after learning. What structure emerges? Can you see the room boundaries in the matrix?

In [ ]:
# ── Exercise 3: SR matrix before vs. after learning ──────────────────────────

# Untrained SR agent (for comparison)
sr_untrained = TDSR(state_size, action_size, lr=0.1, gamma=0.99, poltype="softmax", beta=5.0)
# sr_untrained.M has shape (n_actions, n_states, n_states)
M_before = np.mean(sr_untrained.M, axis=0)
M_after = np.mean(sr_agent_reval.M, axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

n_show = 40
plot_matrix_heatmap(
    M_before[:n_show, :n_show],
    title="SR Matrix BEFORE Learning\n(identity-like: agent expects to stay put)",
    cmap="Blues", ax=ax1, fmt=".1f"
)
ax1.set_xlabel("Future state s'")
ax1.set_ylabel("Current state s")

plot_matrix_heatmap(
    M_after[:n_show, :n_show],
    title="SR Matrix AFTER Learning\n(structure reflects room connectivity)",
    cmap="Blues", ax=ax2, fmt=".1f"
)
ax2.set_xlabel("Future state s'")
ax2.set_ylabel("Current state s")

plt.tight_layout()
plt.show()

## 3.6 Looking Ahead

Key takeaways from this module:

1. **Prediction errors drive learning** in both the brain (dopamine) and in algorithms (TD error)
2. The Rescorla-Wagner rule is a special case of TD(0) — both are about updating predictions based on surprise
3. The **Successor Representation** separates world dynamics from reward, anticipating Active Inference's separation of the B matrix (dynamics) from the C vector (preferences)
4. The RPE framework is powerful but limited: it only computes prediction errors for *reward*. Active Inference computes prediction errors for *all observations*

In Module 4, we will tackle the **exploration-exploitation dilemma** — how should an agent balance seeking reward with seeking information? This is where Active Inference's epistemic value becomes crucial.

**Next:** [Module 4: Exploration, Exploitation, and Model-Free Methods](04_exploration_exploitation.ipynb)

---

## Further Reading

1. **Schultz, W., Dayan, P. & Montague, P.R. (1997).** *A Neural Substrate of Prediction and Reward.* Science, 275, 1593-1599. The landmark paper connecting dopamine to TD errors.

2. **Dayan, P. (1993).** *Improving Generalization for Temporal Difference Learning: The Successor Representation.* Neural Computation, 5(4), 613-624. The original SR paper.

3. **Stachenfeld, K.L., Botvinick, M.M. & Gershman, S.J. (2017).** *The Hippocampus as a Predictive Map.* Nature Neuroscience, 20, 1643-1653. Links the SR to hippocampal place cells.

4. **Rescorla, R.A. & Wagner, A.R. (1972).** *A Theory of Pavlovian Conditioning.* In Black & Prokasy (Eds.), Classical Conditioning II. The foundational prediction error learning rule.

5. **Friston, K.J. et al. (2016).** *Active Inference and Learning.* Neuroscience & Biobehavioral Reviews, 68, 862-879. How prediction errors in AIF relate to (but differ from) the RPE hypothesis.

6. **George, T.M. et al. (2024).** *RatInABox, a toolkit for modelling locomotion and neuronal activity in continuous environments.* eLife. The [successor features demo](https://github.com/TomGeorge1234/RatInABox/blob/main/demos/successor_features_example.ipynb) shows how successor representations emerge in continuous space under different motion policies — a direct extension of our tabular SR analysis.